In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os

print("Libraries loaded")

Libraries loaded


In [2]:
# Cell 2: Load Data
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

gen1 = pd.read_csv('../data/raw/Plant_1_Generation_Data.csv')
weather1 = pd.read_csv('../data/raw/Plant_1_Weather_Sensor_Data.csv')
gen2 = pd.read_csv('../data/raw/Plant_2_Generation_Data.csv')
weather2 = pd.read_csv('../data/raw/Plant_2_Weather_Sensor_Data.csv')

print(f"Plant 1 Gen: {gen1.shape}, Weather: {weather1.shape}")
print(f"Plant 2 Gen: {gen2.shape}, Weather: {weather2.shape}")


LOADING DATA
Plant 1 Gen: (68778, 7), Weather: (3182, 6)
Plant 2 Gen: (67698, 7), Weather: (3259, 6)


In [3]:
# Cell 3: Standardize Date Formats
print("\n" + "="*60)
print("FIXING DATE FORMATS")
print("="*60)

# Plant 1 generation uses dd-mm-yyyy, others use yyyy-mm-dd
gen1['DATE_TIME'] = pd.to_datetime(gen1['DATE_TIME'], dayfirst=True).dt.strftime('%Y-%m-%d %H:%M:%S')
weather1['DATE_TIME'] = pd.to_datetime(weather1['DATE_TIME']).dt.strftime('%Y-%m-%d %H:%M:%S')
gen2['DATE_TIME'] = pd.to_datetime(gen2['DATE_TIME']).dt.strftime('%Y-%m-%d %H:%M:%S')
weather2['DATE_TIME'] = pd.to_datetime(weather2['DATE_TIME']).dt.strftime('%Y-%m-%d %H:%M:%S')

print("Date formats standardized")


FIXING DATE FORMATS
Date formats standardized


In [4]:
# Cell 4: Merge Generation + Weather for Each Plant
print("\n" + "="*60)
print("MERGING GENERATION + WEATHER DATA")
print("="*60)

# Merge Plant 1
df1 = gen1.merge(weather1, on=['DATE_TIME', 'PLANT_ID'], how='inner')
print(f"Plant 1 merged: {df1.shape}")

# Merge Plant 2
df2 = gen2.merge(weather2, on=['DATE_TIME', 'PLANT_ID'], how='inner')
print(f"Plant 2 merged: {df2.shape}")


MERGING GENERATION + WEATHER DATA
Plant 1 merged: (68774, 11)
Plant 2 merged: (67698, 11)


In [5]:
# Cell 5: Combine Both Plants
print("\n" + "="*60)
print("COMBINING BOTH PLANTS")
print("="*60)

df = pd.concat([df1, df2], ignore_index=True)
print(f"Combined dataset: {df.shape}")

# Parse timestamps
df['timestamp'] = pd.to_datetime(df['DATE_TIME'])
df = df.sort_values('timestamp')

print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")



COMBINING BOTH PLANTS
Combined dataset: (136472, 11)
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00
Duration: 33 days


In [6]:
# Cell 6: Handle Missing Values
print("\n" + "="*60)
print("HANDLING MISSING VALUES")
print("="*60)

print(f"Missing values before: {df.isnull().sum().sum()}")

df = df.fillna(method='ffill').fillna(method='bfill')

print(f"Missing values after: {df.isnull().sum().sum()}")


HANDLING MISSING VALUES
Missing values before: 0
Missing values after: 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_13304\3721611204.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill')


In [7]:
# Cell 7: Add Time Features
print("\n" + "="*60)
print("ADDING TIME FEATURES")
print("="*60)

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Add plant indicator (which plant the data is from)
df['is_plant_2'] = (df['PLANT_ID'] == '4136001').astype(int)

print("Time features added: hour, day_of_week, month, is_weekend, is_plant_2")


ADDING TIME FEATURES
Time features added: hour, day_of_week, month, is_weekend, is_plant_2


In [8]:
# Cell 8: Select Features and Target
print("\n" + "="*60)
print("PREPARING FEATURES AND TARGET")
print("="*60)

# Choose AC_POWER as target variable
target_col = 'AC_POWER'

# Drop unnecessary columns
drop_cols = [
    'DATE_TIME', 'PLANT_ID', 'SOURCE_KEY_x', 'SOURCE_KEY_y', 'timestamp',
    'DC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD',
    target_col,
]

# Keep only columns that exist
drop_cols_exist = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols_exist)
y = df[target_col]

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")



PREPARING FEATURES AND TARGET
Features: (136472, 8)
Target: (136472,)

Feature columns: ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_plant_2']


In [9]:
X.to_csv('../data/processed/X_full.csv', index=False)
y.to_csv('../data/processed/y_full.csv', index=False)

In [10]:
# Cell 9: Time-Based Split (70-15-15)
# Calculate split indices based on chronological order
n_samples = len(X)
train_end = int(n_samples * 0.70)   # First 70% of time
val_end = int(n_samples * 0.85)     # Next 15% of time

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

# Print split information
print(f"Train: {len(X_train):,} samples ({len(X_train)/n_samples*100:.1f}%)")
print(f"Val:   {len(X_val):,} samples ({len(X_val)/n_samples*100:.1f}%)")
print(f"Test:  {len(X_test):,} samples ({len(X_test)/n_samples*100:.1f}%)")

# Show date ranges for each split
train_dates = df.iloc[:train_end]['timestamp']
val_dates = df.iloc[train_end:val_end]['timestamp']
test_dates = df.iloc[val_end:]['timestamp']

print(f"\nTrain period: {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Val period:   {val_dates.min().date()} to {val_dates.max().date()}")
print(f"Test period:  {test_dates.min().date()} to {test_dates.max().date()}")


Train: 95,530 samples (70.0%)
Val:   20,471 samples (15.0%)
Test:  20,471 samples (15.0%)

Train period: 2020-05-15 to 2020-06-08
Val period:   2020-06-08 to 2020-06-13
Test period:  2020-06-13 to 2020-06-17


In [11]:
# Cell 10: Scale Features
print("\n" + "="*60)
print("SCALING FEATURES")
print("="*60)

scaler = StandardScaler()

# Fit on training data only
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaling complete (StandardScaler)")
print(f"Train mean: {X_train_scaled.mean().mean():.6f}")
print(f"Train std:  {X_train_scaled.std().mean():.6f}")
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler saved to: ../models/scaler.pkl")


SCALING FEATURES
Scaling complete (StandardScaler)
Train mean: 0.000000
Train std:  0.875005
Scaler saved to: ../models/scaler.pkl


In [12]:
# Cell 11: Save Processed Data
print("\n" + "="*60)
print("SAVING PROCESSED DATA")
print("="*60)

# Create directories
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Save data
X_train_scaled.to_csv('../data/processed/X_train.csv')
y_train.to_csv('../data/processed/y_train.csv')
X_val_scaled.to_csv('../data/processed/X_val.csv')
y_val.to_csv('../data/processed/y_val.csv')
X_test_scaled.to_csv('../data/processed/X_test.csv')
y_test.to_csv('../data/processed/y_test.csv')

print("Files saved to data/processed/")


SAVING PROCESSED DATA
Files saved to data/processed/


In [13]:
# Cell 12: Summary
print("PREPROCESSING COMPLETE")

print(f"Total samples: {len(df):,}")
print(f"Features: {X.shape[1]}")
print(f"Target: {target_col}")
print(f"Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}")
print("\nReady for model training")

PREPROCESSING COMPLETE
Total samples: 136,472
Features: 8
Target: AC_POWER
Train: 95,530, Val: 20,471, Test: 20,471

Ready for model training
